In [ ]:
-- 1. 지난달 결제 매출은 얼마이고, 환불액은 얼마인가

SELECT 
    SUM(p.amount) - SUM(IFNULL(c.discount_amount, 0)) - SUM(IFNULL(r.refund_amount, 0)) AS 매출,
    SUM(r.refund_amount) AS 환불액
FROM tb_payment p
    LEFT JOIN tb_refund r ON p.id = r.payment_id
    LEFT JOIN tb_coupon c ON p.id = c.payment_id


In [ ]:
-- 2. 강의별 수강 완료율(진도 100% 도달 비율)은 얼마인가

SELECT 
    lecture_id,
    COUNT(CASE WHEN completion_rate = 100 THEN 1 ELSE 0 END) / COUNT(*) AS 완료율
FROM tb_enrollment
GROUP BY lecture_id

In [ ]:
-- 3. 강사별 누적 수강생 수와 평균 별점은 얼마인가

SELECT 
    lr.lecturer_id,
    lr.lecturer_name,
    COUNT(DISTINCT l.id) AS 누적_수강생_수,
    AVG(r.rating) AS 평균_별점
FROM tb_lecturers lr
    LEFT JOIN tb_lectures l ON lr.lecturer_id = l.lecturer_id2
    LEFT JOIN tb_review r ON l.id = r.lecture_id
GROUP BY lr.lecturer_id, lr.lecturer_name

In [ ]:
-- 4. 쿠폰을 쓴 수강생과 쓰지 않은 수강생의 평균 결제액 차이는 얼마인가

SELECT 
    m.member_id,
    AVG(CASE WHEN c.id IS NOT NULL THEN p.amount ELSE NULL END) AS 쿠폰_사용_평균_결제액,
    AVG(CASE WHEN c.id IS NULL THEN p.amount ELSE NULL END) AS 쿠폰_미사용_평균_결제액
FROM tb_members m
    LEFT JOIN tb_payment p ON m.member_id = p.member_id
    LEFT JOIN tb_coupon c ON p.id = c.payment_id
GROUP BY m.member_id

In [ ]:
-- 5. 어느 카테고리의 강의가 중도 이탈(진도 20% 미만)이 많은가

SELECT 
    c.category_id,
    c.category_name,
    COUNT(e.member_id) AS 중도_이탈_수
FROM tb_category c
    LEFT JOIN tb_lectures l on c.category_id = l.category_id
    LEFT JOIN tb_enrollment e on l.id = e.lecture_id
WHERE e.completion_rate < 20
GROUP BY c.category_id, c.category_name
ORDER BY 중도_이탈_수 DESC;

In [ ]:
-- 6. 한 번도 수강 신청이 없는 강의는 무엇인가

SELECT l.id, l.lecture_name
FROM tb_lectures l
    LEFT JOIN tb_enrollment e ON l.id = e.lecture_id
WHERE e.lecture_id IS NULL;

In [ ]:
-- 7. 문의가 3건 이상 들어온 강의는 무엇인가

SELECT l.id, l.lecture_name
FROM tb_lectures l
    LEFT JOIN tb_review r ON l.id = r.lecture_id
WHERE r.id IS NOT NULL
GROUP BY l.id, l.lecture_name
HAVING COUNT(r.id) >= 3;

In [ ]:
-- 8. 재수강(같은 회원이 두 개 이상 강의를 결제)한 회원의 비율은 얼마인가

SELECT 
    COUNT(*) * 100.0 / (SELECT COUNT(DISTINCT member_id) FROM tb_enrollment) AS 재수강_회원_비율
FROM (
    SELECT member_id
    FROM tb_enrollment
    GROUP BY member_id
    HAVING COUNT(DISTINCT lecture_id) > 1
);